In [ ]:
import os
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from datasets import Dataset


tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
model = GPT2LMHeadModel.from_pretrained("gpt2-medium", torch_dtype=torch.bfloat16)
tokenizer.pad_token = tokenizer.eos_token


data = []
with open(os.path.join("E2E", "train.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
train_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "valid.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
valid_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "test.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
test_dataset = Dataset.from_list(data)

In [2]:
max_seq_len = 128

tokenized_data = []
for example in train_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_train_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in valid_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_valid_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in test_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_test_dataset = Dataset.from_list(tokenized_data)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2_e2e_model",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="epoch",
    save_steps=1,
    save_total_limit=5,
    dataloader_drop_last=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_train_dataset,
    eval_dataset=preprocessed_valid_dataset
)

trainer.train()

In [ ]:
list_test = []

for i in range(len(test_dataset)):
    if len(list_test) == 0 or list_test[-1]['context'] != test_dataset[i]['context']:
        list_test.append({'context': test_dataset[i]['context'], 'references': [test_dataset[i]['completion']]})
    else:
        list_test[-1]['references'].append(test_dataset[i]['completion'])

print(len(list_test))

In [ ]:
model = model.to('cuda')
model.eval()

for i in range(len(list_test)):
    encoded = tokenizer(list_test[i]['context'] + tokenizer.bos_token, return_tensors='pt').to('cuda')
    output = model.generate(
        input_ids=encoded['input_ids'],
        attention_mask=encoded['attention_mask'],
        max_length=128,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    output = tokenizer.decode(output[0])
    list_test[i]['prediction'] = output[len(list_test[i]['context']):]

for i in range(len(list_test)):
    print(list_test[i]['prediction'])

In [ ]:
print(list_test[0]['context'])
print(list_test[0]['references'])
print(list_test[0]['prediction'])
print(list_test[1]['context'])
print(list_test[1]['references'])
print(list_test[1]['prediction'])

In [ ]:
list_references = []
list_prediction = []

for i in range(len(list_test)):
    list_references.append(list_test[i]['references'])
    list_prediction.append(list_test[i]['prediction'][len(tokenizer.bos_token)+1:-len(tokenizer.eos_token)])

print(len(list_references))
print(len(list_prediction))

In [ ]:
print(list_references[0])
print(list_prediction[0])

cnt = 0
for txt in list_prediction:
    if '/n/n' in txt:
        cnt += 1
print(cnt)

In [9]:
def save_benchmark_data(list_references, list_prediction, reference_file, prediction_file):
    with open(reference_file, "w", encoding="utf-8") as ref_file:
        for refs in list_references:
            for ref in refs:
                ref_file.write(ref + "\n")
            ref_file.write("\n")

    with open(prediction_file, "w", encoding="utf-8") as pred_file:
        for pred in list_prediction:
            pred_file.write(pred + "\n")

reference_file = "references.txt"
prediction_file = "predictions.txt"

save_benchmark_data(list_references, list_prediction, reference_file, prediction_file)